In [ ]:
import pandas as pd
import numpy as np

# Load cleaned dataset
df = pd.read_csv("E:\Harsha_Masters\hed\Study_2\Data Science Club\Muti_Layered Fraud Detection\Multi_Layered_Behavioral_Fraud_Detection\Multi-Layered-Behavioral-Fraud-Detection\data\data_clean.csv.csv")
df.head()


# 1. Core Behavioural Features (Transaction Intensity)

In [ ]:
# Transaction intensity (raw)
df['txn_intensity'] = df['num_transactions']

# Transactions per day (requires account_age_days)
df['txn_per_day'] = df['num_transactions'] / df['account_age_days']

# Burstiness: many transactions in few login days
df['txn_burstiness'] = df['num_transactions'] / (df['last_login_days'] + 1)

# Login frequency
df['login_frequency'] = 1 / (df['last_login_days'] + 1)


# 2. Velocity and Ratio Features

In [ ]:
# Spend velocity
df['spend_velocity'] = df['monthly_spend'] / df['num_transactions']

# Ratio of average to total spend
df['avg_vs_total_ratio'] = df['avg_transaction_value'] / (df['total_spend'] + 1)

# 3. Atypicality Features (Z‑Scores by Group)

In [ ]:
# Payment‑method z‑scores

df['z_num_txn_payment'] = df.groupby('payment_method')['num_transactions'].transform(
    lambda x: (x - x.mean()) / x.std()
)

df['z_monthly_spend_payment'] = df.groupby('payment_method')['monthly_spend'].transform(
    lambda x: (x - x.mean()) / x.std()
)


# Country‑level z‑scores

df['z_num_txn_country'] = df.groupby('country')['num_transactions'].transform(
    lambda x: (x - x.mean()) / x.std()
)

df['z_monthly_spend_country'] = df.groupby('country')['monthly_spend'].transform(
    lambda x: (x - x.mean()) / x.std()
)


# 4
. Keep the Validated Rule‑Based Feature

In [ ]:
# suspicion_score already exists and is validated
df['suspicion_score'] = df['suspicion_score']

# 7. Minimal Encoding for Weak Categorical Features

In [ ]:
# Gender (binary)
df['gender_binary'] = df['gender'].map({'Male': 1, 'Female': 0})

# Country (light one-hot)
country_dummies = pd.get_dummies(df['country'], prefix='country', drop_first=True)
df = pd.concat([df, country_dummies], axis=1)

# Payment method (light one-hot)
pm_dummies = pd.get_dummies(df['payment_method'], prefix='pm', drop_first=True)
df = pd.concat([df, pm_dummies], axis=1)


# 8. Interaction Features (Only with num_transactions)

In [ ]:
df['txn_x_spend'] = df['num_transactions'] * df['monthly_spend']
df['txn_x_login'] = df['num_transactions'] / (df['last_login_days'] + 1)

# 9. Drop Redundant or Low‑Value Columns

In [ ]:
cols_to_drop = [
    'avg_transaction_value',   # weak signal
    'gender',                  # replaced by gender_binary
    'country',                 # encoded
    'payment_method'           # encoded
]

df = df.drop(columns=cols_to_drop)


# 10. Final Checks and Save Engineered Dataset

In [ ]:
df.info()
df.describe().T


In [ ]:
df.to_csv("data_engineered.csv", index=False)
print("Saved as data_engineered.csv")